In [ ]:
import os, sys, time, subprocess
def W(p, msg):
    with open(p, 'a') as f: f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    print(msg, flush=True)
TR = '/kaggle/working/trace.log'
W(TR, '=== nb164 START ===')
W(TR, f'python={sys.version[:30]}')
import torch
W(TR, f'torch={torch.__version__} cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    W(TR, f'device={torch.cuda.get_device_name(0)} cc={torch.cuda.get_device_capability(0)}')

In [ ]:
# Install boltz via system pip (no --target, no version pins)
# Use --no-deps to avoid transformers conflict, then install rest manually
W(TR, '=== INSTALL: try system pip boltz ===')
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'boltz'],
                   capture_output=True, text=True, timeout=1800)
W(TR, f'install rc={r.returncode} elapsed={time.time()-t0:.0f}s')
if r.returncode != 0:
    W(TR, f'stderr last 2000: {r.stderr[-2000:]}')
else:
    W(TR, 'install OK')

In [ ]:
W(TR, '=== boltz --help ===')
r = subprocess.run(['boltz', '--help'], capture_output=True, text=True, timeout=120)
W(TR, f'rc={r.returncode}')
W(TR, f'stdout[:800]: {r.stdout[:800]}')
if r.returncode != 0:
    W(TR, f'stderr last 1500: {r.stderr[-1500:]}')
W(TR, '=== nb164 helpcheck END ===')

In [ ]:
from pathlib import Path
PXR_SEQ = 'LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCSIVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWEVLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELNGLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWANKTKDPLLLEAHALDQFSCK'
yaml_str = f'version: 1\nsequences:\n- protein:\n    id: A\n    sequence: {PXR_SEQ}\n- ligand:\n    id: B\n    smiles: CCOc1ccccc1\nproperties:\n- affinity:\n    binder: B\n'
Path('/kaggle/working/test.yaml').write_text(yaml_str)
OUT = Path('/kaggle/working/o_test'); OUT.mkdir(exist_ok=True)
cmd = ['boltz', 'predict', '/kaggle/working/test.yaml', '--out_dir', str(OUT),
       '--use_msa_server', '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
W(TR, f'=== PREDICT: {" ".join(cmd)}')
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True, timeout=2400)
W(TR, f'predict rc={r.returncode} elapsed={(time.time()-t0)/60:.1f}min')
W(TR, f'stdout tail 2000: {r.stdout[-2000:]}')
if r.returncode != 0:
    W(TR, f'stderr tail 2000: {r.stderr[-2000:]}')
import json
aff_files = list(OUT.rglob('*affinity*.json'))
W(TR, f'affinity files: {len(aff_files)}')
for jf in aff_files:
    W(TR, f'{jf.name}: {open(jf).read()[:500]}')
W(TR, f'all files in OUT: {[str(p.relative_to(OUT)) for p in OUT.rglob("*")][:30]}')
W(TR, '=== nb164 END ===')